In [1]:
!pip install scikit-learn pandas joblib pyarrow

import boto3
import pandas as pd
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

s3 = boto3.client("s3")
bucket = "propelnoi-vijay-datalake"

# find training CSV
resp = s3.list_objects_v2(Bucket=bucket, Prefix="modeling/maintenance_rf/train/")
train_files = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".csv")]
train_key = train_files[0]

resp = s3.list_objects_v2(Bucket=bucket, Prefix="modeling/maintenance_rf/validation/")
val_files = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".csv")]
val_key = val_files[0]

local_train = "/tmp/maintenance_train.csv"
local_val = "/tmp/maintenance_val.csv"

s3.download_file(bucket, train_key, local_train)
s3.download_file(bucket, val_key, local_val)

train_df = pd.read_csv(local_train, header=None)
val_df = pd.read_csv(local_val, header=None)

# target first
X_train = train_df.iloc[:, 1:]
y_train = train_df.iloc[:, 0]

X_val = val_df.iloc[:, 1:]
y_val = val_df.iloc[:, 0]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

preds = model.predict(X_val)
probs = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, preds))
print(confusion_matrix(y_val, preds))
print(classification_report(y_val, preds))

Accuracy: 1.0
[[2935    0]
 [   0 1439]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2935
           1       1.00      1.00      1.00      1439

    accuracy                           1.00      4374
   macro avg       1.00      1.00      1.00      4374
weighted avg       1.00      1.00      1.00      4374



In [2]:
# Save model artifact to S3
local_model = "/tmp/maintenance_rf_model.joblib"
joblib.dump(model, local_model)

s3.upload_file(
    local_model,
    bucket,
    "model-artifacts/maintenance_rf/maintenance_rf_model.joblib"
)

print("Uploaded model artifact to S3")

Uploaded model artifact to S3


In [4]:
# ============================================================
# Predictive Maintenance Shield - Full Corrected Notebook
# Purpose:
# 1. Load trained Random Forest model artifact from S3
# 2. Load inference CSV from S3
# 3. Load ALL metadata rows from Athena using paginator
# 4. Score predicted maintenance risk
# 5. Create final output_maintenance_risk_predictions dataset
# 6. Upload final CSV to S3
# ============================================================

# ------------------------------------------------------------
# 0. Install packages if needed
# ------------------------------------------------------------
# Run once if needed:
# !pip install scikit-learn pandas joblib pyarrow

import os
import time
import boto3
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
BUCKET = "propelnoi-vijay-datalake"
DATABASE = "propelnoi_db"

MODEL_KEY = "model-artifacts/maintenance_rf/maintenance_rf_model.joblib"
INFERENCE_PREFIX = "modeling/maintenance_rf/inference_input/"
ATHENA_OUTPUT = f"s3://{BUCKET}/outputs/athenaresults/"
FINAL_OUTPUT_KEY = "outputs/output_maintenance_risk_predictions/output_maintenance_risk_predictions.csv"

# ------------------------------------------------------------
# 2. AWS clients
# ------------------------------------------------------------
s3 = boto3.client("s3")
athena = boto3.client("athena")

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------
def wait_for_athena(query_execution_id: str, poll_seconds: int = 2) -> None:
    """Wait for Athena query to finish."""
    while True:
        response = athena.get_query_execution(QueryExecutionId=query_execution_id)
        state = response["QueryExecution"]["Status"]["State"]

        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            if state != "SUCCEEDED":
                reason = response["QueryExecution"]["Status"].get("StateChangeReason", "Unknown Athena failure")
                raise RuntimeError(f"Athena query failed: {state} - {reason}")
            break

        time.sleep(poll_seconds)


def run_athena_query(query: str, database: str = DATABASE, output_location: str = ATHENA_OUTPUT) -> pd.DataFrame:
    """
    Run Athena query and return ALL rows as pandas DataFrame.
    Uses paginator so results are not truncated at ~1000 rows.
    """
    start_response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output_location},
    )

    query_execution_id = start_response["QueryExecutionId"]
    wait_for_athena(query_execution_id)

    paginator = athena.get_paginator("get_query_results")
    page_iterator = paginator.paginate(QueryExecutionId=query_execution_id)

    headers = None
    rows_out = []

    for page in page_iterator:
        rows = page["ResultSet"]["Rows"]

        if headers is None:
            headers = [c.get("VarCharValue", "") for c in rows[0]["Data"]]
            rows = rows[1:]  # skip header row only once

        for row in rows:
            vals = [c.get("VarCharValue", None) for c in row["Data"]]
            if len(vals) < len(headers):
                vals.extend([None] * (len(headers) - len(vals)))
            rows_out.append(vals)

    df = pd.DataFrame(rows_out, columns=headers if headers else [])
    return df


def list_s3_csv_files(bucket: str, prefix: str) -> list[str]:
    """List CSV files in S3 prefix."""
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    files = [
        obj["Key"]
        for obj in response.get("Contents", [])
        if obj["Key"].endswith(".csv")
    ]
    return files


def download_s3_file(bucket: str, key: str, local_path: str) -> None:
    """Download a file from S3."""
    s3.download_file(bucket, key, local_path)


def upload_s3_file(local_path: str, bucket: str, key: str) -> None:
    """Upload a file to S3."""
    s3.upload_file(local_path, bucket, key)


# ------------------------------------------------------------
# 4. Load trained Random Forest model
# ------------------------------------------------------------
print("Downloading trained Random Forest model artifact from S3...")
local_model_path = "/tmp/maintenance_rf_model.joblib"
download_s3_file(BUCKET, MODEL_KEY, local_model_path)

print("Loading model...")
model = joblib.load(local_model_path)
print("Model loaded successfully.")

# ------------------------------------------------------------
# 5. Load inference CSV
# ------------------------------------------------------------
print("Locating inference CSV in S3...")
csv_files = sorted(list_s3_csv_files(BUCKET, INFERENCE_PREFIX))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in s3://{BUCKET}/{INFERENCE_PREFIX}")

# Use latest file if multiple ever appear
inference_key = csv_files[-1]
print("Using inference file:", inference_key)

local_infer_csv = "/tmp/maintenance_inference.csv"
download_s3_file(BUCKET, inference_key, local_infer_csv)

print("Reading inference CSV...")
X = pd.read_csv(local_infer_csv, header=None)
print("Inference rows:", len(X))
print("Inference shape:", X.shape)

# ------------------------------------------------------------
# 6. Load scoring metadata from Athena (ALL rows)
# ------------------------------------------------------------
print("Querying scoring metadata from Athena...")
metadata_query = """
SELECT
    property_id,
    CAST(month AS VARCHAR) AS month,
    city,
    property_type,
    units,
    system_type,
    repair_cost,
    asset_age,
    avg_repair_cost_3m,
    risk_score
FROM propelnoi_db.output_maintenance_scoring_input
ORDER BY property_id, system_type, month
"""

meta_df = run_athena_query(metadata_query)

if meta_df.empty:
    raise ValueError("Athena returned no rows from output_maintenance_scoring_input")

# Convert numeric columns
numeric_cols = ["units", "repair_cost", "asset_age", "avg_repair_cost_3m", "risk_score"]
for c in numeric_cols:
    if c in meta_df.columns:
        meta_df[c] = pd.to_numeric(meta_df[c], errors="coerce")

print("Metadata rows:", len(meta_df))
print("Metadata shape:", meta_df.shape)

# Safety check
if len(X) != len(meta_df):
    raise ValueError(
        f"Row mismatch: inference rows={len(X)} vs metadata rows={len(meta_df)}"
    )

# ------------------------------------------------------------
# 7. Score predictions
# ------------------------------------------------------------
print("Scoring maintenance risk predictions...")
pred_class = model.predict(X)
pred_prob = model.predict_proba(X)[:, 1]

meta_df["predicted_probability"] = pred_prob
meta_df["predicted_label"] = pred_class

# ------------------------------------------------------------
# 8. Add business logic
# ------------------------------------------------------------
print("Calculating predicted risk score, band, cost impact, alert flag...")

# Convert probability to 0-100 score
meta_df["predicted_risk_score"] = (meta_df["predicted_probability"] * 100).round(2)

def risk_band(score: float) -> str:
    if pd.isna(score):
        return "UNKNOWN"
    if score >= 75:
        return "HIGH"
    if score >= 50:
        return "MEDIUM"
    return "LOW"

meta_df["predicted_risk_band"] = meta_df["predicted_risk_score"].apply(risk_band)

# Estimated cost impact (simple hackathon formula)
meta_df["estimated_cost_impact"] = (
    meta_df["repair_cost"].fillna(0.0) *
    (1 + meta_df["predicted_probability"].fillna(0.0))
).round(2)

meta_df["alert_flag"] = np.where(meta_df["predicted_risk_band"] == "HIGH", "Y", "N")

# ------------------------------------------------------------
# 9. Build final output dataframe
# ------------------------------------------------------------
output_df = meta_df[
    [
        "property_id",
        "month",
        "city",
        "property_type",
        "system_type",
        "repair_cost",
        "asset_age",
        "avg_repair_cost_3m",
        "predicted_probability",
        "predicted_risk_score",
        "predicted_risk_band",
        "estimated_cost_impact",
        "alert_flag"
    ]
].copy()

# Optional rounding
for c in ["repair_cost", "avg_repair_cost_3m", "predicted_probability", "predicted_risk_score", "estimated_cost_impact"]:
    output_df[c] = pd.to_numeric(output_df[c], errors="coerce").round(4)

# Keep month as STRING for Athena CSV compatibility
output_df["month"] = pd.to_datetime(output_df["month"], errors="coerce").dt.strftime("%Y-%m-%d")

print("Final output preview:")
display(output_df.head(10))

# ------------------------------------------------------------
# 10. Save locally
# ------------------------------------------------------------
local_output_csv = "/tmp/output_maintenance_risk_predictions.csv"
output_df.to_csv(local_output_csv, index=False)
print("Saved local output to:", local_output_csv)

# ------------------------------------------------------------
# 11. Upload to S3
# ------------------------------------------------------------
print("Uploading final maintenance predictions to S3...")
upload_s3_file(local_output_csv, BUCKET, FINAL_OUTPUT_KEY)

print(f"Uploaded to s3://{BUCKET}/{FINAL_OUTPUT_KEY}")

# ------------------------------------------------------------
# 12. Validation summary
# ------------------------------------------------------------
print("\nSummary stats:")
print("Rows:", len(output_df))
print("Cities:", output_df["city"].nunique())
print("Risk band distribution:")
print(output_df["predicted_risk_band"].value_counts(dropna=False))
print("\nAlert distribution:")
print(output_df["alert_flag"].value_counts(dropna=False))



Loading model...
Model loaded successfully.
Locating inference CSV in S3...
Using inference file: modeling/maintenance_rf/inference_input/part-00000-59d350c9-a665-4a27-ad5b-1287722425b7-c000.csv
Reading inference CSV...
Inference rows: 3684
Inference shape: (3684, 8)
Querying scoring metadata from Athena...
Metadata rows: 3684
Metadata shape: (3684, 10)
Scoring maintenance risk predictions...
Calculating predicted risk score, band, cost impact, alert flag...
Final output preview:


,property_id,month,city,property_type,system_type,repair_cost,asset_age,avg_repair_cost_3m,predicted_probability,predicted_risk_score,predicted_risk_band,estimated_cost_impact,alert_flag
0,P00001,2024-11-01,Seattle,Mixed Use,Electrical,4419.43,19,3769.6567,0.0005,0.05,LOW,4421.65,N
1,P00001,2025-06-01,Seattle,Mixed Use,Elevator,4333.24,17,3032.6567,0.0119,1.19,LOW,4384.62,N
2,P00001,2025-12-01,Seattle,Mixed Use,Fire System,4049.64,21,4075.8233,0.0076,0.76,LOW,4080.43,N
3,P00001,2025-08-01,Seattle,Mixed Use,HVAC,5746.19,17,5386.4567,0.0001,0.01,LOW,5746.58,N
4,P00001,2025-02-01,Seattle,Mixed Use,Plumbing,4770.45,17,4536.7367,0.0000,0.00,LOW,4770.45,N
5,P00001,2025-11-01,Seattle,Mixed Use,Roof,4870.19,22,4721.3267,0.9765,97.65,HIGH,9626.05,Y
6,P00002,2025-08-01,Chicago,Mixed Use,Electrical,1401.95,8,1327.2100,0.0001,0.01,LOW,1402.05,N
7,P00002,2025-05-01,Chicago,Mixed Use,Elevator,3078.93,14,2159.3200,0.0016,0.16,LOW,3083.73,N
8,P00002,2025-10-01,Chicago,Mixed Use,Fire System,3559.82,8,3813.9333,0.0023,0.23,LOW,3568.09,N
9,P00002,2024-02-01,Chicago,Mixed Use,HVAC,1348.63,14,2164.9133,0.9990,99.90,HIGH,2695.93,Y


Saved local output to: /tmp/output_maintenance_risk_predictions.csv
Uploading final maintenance predictions to S3...
Uploaded to s3://propelnoi-vijay-datalake/outputs/output_maintenance_risk_predictions/output_maintenance_risk_predictions.csv

Summary stats:
Rows: 3684
Cities: 6
Risk band distribution:
predicted_risk_band
LOW     2535
HIGH    1149
Name: count, dtype: int64

Alert distribution:
alert_flag
N    2535
Y    1149
Name: count, dtype: int64
